# 🧪 W6-D5 概念实验：Agent 开发实战与框架设计

> 配套阅读：`第6周-Day5-Agent开发实战与框架设计.md`（框架三要素、选型决策树在那边）
>
> 玩具 Agent 和生产 Agent 的差距在框架层。本 notebook 用可运行代码回答：
> 1. **错误处理与重试**：指数退避重试把成功率从多少拉到多少？代价是什么？
> 2. **状态管理**：进程崩溃后，无状态 / checkpoint / 事件溯源 三种策略要重做多少工作？
> 3. **可观测性**：结构化 trace 长什么样？怎么用它定位慢步骤？
> 4. **选型**：重试策略与状态策略组合出什么形态？

环境：仅 numpy / 标准库 / matplotlib；时间用虚拟时钟模拟，不真实 sleep。

## 实验 1：带重试的工具执行器（指数退避 + 抖动）

生产工具会失败（超时、限流、网络抖动）。实现一个 `call_with_retry`：
失败按 2^n 秒指数退避 + 随机抖动重试。用不稳定的模拟工具统计
**最终成功率 / 平均调用次数 / 总等待时间**，对比 不重试 / 固定间隔 / 指数退避。

In [ ]:
import numpy as np
rng = np.random.default_rng(5)

class VirtualClock:            # 虚拟时钟：不真实 sleep，只记账
    t = 0.0
    @classmethod
    def wait(cls, sec): cls.t += sec

def flaky_call(fail_p=0.35):
    """模拟不稳定工具：35% 失败率，但连续调用相互独立。"""
    return rng.random() >= fail_p

def call_with_retry(max_retries=4, backoff="exp"):
    """返回 (success, n_calls, total_wait)。exp=指数退避+jitter，fixed=固定1s，none=不重试"""
    n, wait_total = 1, 0.0
    if flaky_call():                       # 首次就成功
        return True, n, wait_total
    if backoff == "none":
        return False, n, wait_total
    for attempt in range(1, max_retries + 1):
        delay = (2 ** attempt if backoff == "exp" else 1.0) * rng.uniform(0.8, 1.2)
        VirtualClock.wait(delay); wait_total += delay; n += 1
        if flaky_call():
            return True, n, wait_total
    return False, n, wait_total

N = 3000
print(f"单次失败率 35% 的工具，{N} 次任务：")
print(f"{'策略':<12}{'最终成功率':>10}{'平均调用数':>10}{'平均等待(s)':>12}")
stats = {}
for policy in ["none", "fixed", "exp"]:
    res = [call_with_retry(backoff=policy) for _ in range(N)]
    succ = np.mean([r[0] for r in res])
    calls = np.mean([r[1] for r in res])
    wait = np.mean([r[2] for r in res])
    stats[policy] = (succ, calls, wait)
    print(f"{policy:<12}{succ:>10.1%}{calls:>10.2f}{wait:>12.1f}")
print("\n解读：重试把成功率从 65% 拉到 ~99%，代价是尾部延迟变长（重试要等）；")
print("指数退避+抖动 对'下游正在过载'最友好——错峰重试，避免雪上加霜。")

## 实验 2：崩溃恢复 —— 三种状态管理策略的工作量账

Agent 任务 10 步。随机在第 k 步崩溃（模拟 200 次），对比恢复成本：
- **无持久化**：从头重跑 → 浪费 k-1 步
- **逐步 checkpoint**：从上一步快照恢复 → 浪费 ≤1 步，但每步要写快照
- **事件溯源**：只追加事件日志，恢复=重放事件（重放比执行便宜，如只是状态更新）

In [ ]:
rng = np.random.default_rng(9)
N_RUNS, N_STEPS = 200, 10

def crash_run(strategy):
    """返回 (浪费的执行步数, 持久化开销次数)。崩溃点均匀分布。"""
    crash_at = rng.integers(1, N_STEPS + 1)          # 在第 crash_at 步前崩溃
    if strategy == "none":
        return crash_at - 1, 0                        # 已做的步全部白做
    if strategy == "checkpoint":
        return min(1, crash_at - 1), crash_at - 1     # 最多重做1步；每步都写快照
    return 0, crash_at - 1                            # 事件溯源：重放日志，不重执行工具
    # 简化假设：工具调用不可缓存；真实系统里事件溯源还能配合结果缓存进一步省

res = {}
for s in ["none", "checkpoint", "event_sourcing"]:
    runs = [crash_run(s) for _ in range(N_RUNS)]
    wasted = np.mean([r[0] for r in runs])
    persist = np.mean([r[1] for r in runs])
    res[s] = (wasted, persist)
    print(f"{s:<16} 平均浪费执行 {wasted:.2f} 步 | 平均持久化写入 {persist:.1f} 次")
print("\n解读：无持久化在长任务上浪费巨大；checkpoint 用写入换重做；")
print("事件溯源写入同量级但可完整重建历史（审计+回放调试都靠它）——框架的核心选择。")

## 实验 3：可观测性 —— 结构化 Trace

框架要替你回答："这次 Agent 运行为什么慢/为什么错？"
实现 `Tracer`：每步记录 tool/状态/耗时/token，运行一个 3 工具任务，
输出 JSONL 日志 + 汇总统计（总耗时、慢步骤定位、失败率）。

In [ ]:
import json, time

class Tracer:
    def __init__(self): self.events = []
    def log(self, step, tool, status, ms, tokens):
        self.events.append(dict(step=step, tool=tool, status=status, ms=ms, tokens=tokens))
    def jsonl(self):
        return "\n".join(json.dumps(e, ensure_ascii=False) for e in self.events)

tracer = Tracer()
plan = [  # 模拟一次 Agent 运行的四个阶段（含一次失败重试）
    ("llm_decide", "ok", 820, 460),
    ("query_inventory", "ok", 150, 0),
    ("query_order", "timeout_retry", 3000, 0),
    ("query_order", "ok", 180, 0),
    ("llm_summarize", "ok", 950, 520),
]
for i, (tool, status, ms, tok) in enumerate(plan, 1):
    tracer.log(i, tool, status, ms, tok)

print(tracer.jsonl())
ms = [e["ms"] for e in tracer.events]
tokens = [e["tokens"] for e in tracer.events]
fails = sum(1 for e in tracer.events if e["status"] != "ok")
slowest = max(tracer.events, key=lambda e: e["ms"])
print(f"\n汇总: 总耗时 {sum(ms)/1000:.2f}s | 总token {sum(tokens)} | 异常事件 {fails}")
print(f"最慢步骤: {slowest['tool']}（{slowest['ms']}ms）→ 优化优先级第一名")
print("解读：结构化 trace 让'哪步慢/哪步错'变成可查询数据，而不是靠翻日志猜。")

## 实验 4：可视化 —— 重试策略与崩溃恢复成本

In [ ]:
# matplotlib 中文字体配置（NotoSansCJK，每次画图前先跑这段）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = fontManager_font = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
labels = {"none": "不重试", "fixed": "固定间隔重试", "exp": "指数退避+抖动"}

# 左：成功率 vs 平均等待（重试策略）
names = list(stats)
x = np.arange(len(names))
axes[0].bar(x - 0.2, [stats[n][0] * 100 for n in names], 0.4, color="#2a9d8f", label="成功率%")
axes[0].bar(x + 0.2, [stats[n][2] / 10 for n in names], 0.4, color="#e76f51", label="平均等待/10s")
axes[0].set_xticks(x); axes[0].set_xticklabels([labels[n] for n in names], fontsize=9)
axes[0].set_title("重试策略：成功率 ↑ vs 尾部延迟 ↑")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, axis="y")

# 右：崩溃恢复的浪费步数（状态策略）
s_labels = {"none": "无持久化", "checkpoint": "逐步checkpoint", "event_sourcing": "事件溯源"}
names2 = list(res)
axes[1].bar(np.arange(3), [res[n][0] for n in names2], 0.5, color="#219ebc")
axes[1].set_xticks(np.arange(3)); axes[1].set_xticklabels([s_labels[n] for n in names2], fontsize=9)
axes[1].set_ylabel("崩溃后平均浪费执行步数")
axes[1].set_title("10 步任务随机崩溃：恢复成本对比")
axes[1].grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()
print("框架设计的本质：用'写入/等待'的确定性成本，换'崩溃/失败'的不确定性损失。")
print("选型顺序建议：先可观测(trace) → 再重试 → 最后状态策略（按痛点顺序补齐）。")

## 小结

- 重试+指数退避+抖动：成功率 65%→99%，但接受更长的尾部延迟
- 状态策略是"写入 vs 重做"的权衡；事件溯源额外赠送审计与回放调试
- 可观测性先行：没有 trace，其他优化都是盲打
- 这些组件合起来 = md 里的"自研轻量框架"，也是选型评估 LangChain 等的检查清单